# DAA Lab 1 — Comparative Study of Algorithm Efficiency and Performance Analysis

| Field | Details |
|---|---|
| **Student Name** | RAKESH G |
| **Roll Number** | 2401201064 |
| **Course** | BCA (AI & DS) |
| **Semester** | 5th Semester |
| **University** | K.R. Mangalam University |
| **Subject** | Design and Analysis of Algorithms |
| **Lab** | Lab 1 |

---

## Section 1 · Introduction

Algorithm analysis is one of the most foundational skills in computer science.  Two programs that both correctly sort a list of numbers can differ by a factor of **hundreds** in actual running time when the list is large.  The goal of this lab is to make that difference concrete — to measure it with real timings and real memory readings rather than relying solely on Big-O notation.

This notebook covers:

* Four classical **sorting algorithms** implemented from scratch
* Three approaches to computing **Fibonacci numbers**
* Theoretical complexity analysis (pseudocode, Big-O, advantages/limitations)
* **Experimental performance analysis** — execution time, memory usage, comparison counts
* **Visualizations** connecting measured data to theoretical predictions
* **Discussion** of scalability, time-space trade-offs, and when to use each algorithm

---

## Section 2 · Objectives

1. Implement four sorting algorithms (Bubble Sort, Insertion Sort, Merge Sort, Quick Sort) **without using Python's built-in sort**.
2. Implement three Fibonacci algorithms (Recursive, Iterative, Dynamic Programming).
3. Measure execution time, peak memory usage, and number of comparisons for each algorithm.
4. Compare measured performance against theoretical Big-O predictions.
5. Identify the most suitable algorithm for different data sizes and conditions.
6. Discuss time-space trade-offs and scalability implications.

---

## Section 3 · Environment and Library Setup

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import os
import sys
import time
import random
import tracemalloc
import statistics

# ── Third-party ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# ── Make sure project root is importable ──────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Project modules ───────────────────────────────────────────────────────
from algorithms.sorting import bubble_sort, insertion_sort, merge_sort, quick_sort
from algorithms.fibonacci import fib_recursive, fib_iterative, fib_dp
from analysis.performance import (
    measure_time_and_memory,
    build_sorting_datasets,
    run_sorting_experiments,
    run_fibonacci_experiments,
)
from analysis.visualizations import generate_all_plots

# ── Notebook display settings ─────────────────────────────────────────────
pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
plt.rcParams['figure.dpi'] = 100

print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print("\nAll imports successful ✓")

---

## Section 4 · Sorting Algorithms — Theory and Implementation

### 4.1 Bubble Sort

**How it works:** Bubble Sort makes repeated passes through the list. In each pass it compares every adjacent pair and swaps them if they are in the wrong order. After each pass the largest unsorted element has "bubbled" to its correct position at the right end. An `swapped` flag enables **early termination**: if a full pass completes with no swaps the list is already sorted.

**Pseudocode:**
```
ALGORITHM BubbleSort(A, n)
  FOR outer = 0 TO n-2
    made_a_swap = FALSE
    FOR inner = 0 TO n-2-outer
      comparisons += 1
      IF A[inner] > A[inner+1]
        SWAP A[inner], A[inner+1]
        made_a_swap = TRUE
    IF made_a_swap == FALSE
      BREAK            ← already sorted, exit early
  RETURN A
```

**Complexity Table:**

| Case | Time | Space |
|---|---|---|
| Best (already sorted) | O(n) | O(1) |
| Average | O(n²) | O(1) |
| Worst (reverse sorted) | O(n²) | O(1) |

**Advantages:** Very simple to write and explain; stable; adaptive on nearly-sorted data.

**Limitations:** Very slow for large datasets — quadratic time in the average and worst cases.

**Best used when:** Teaching purposes, tiny lists (n < 20), or nearly-sorted data where the early-exit flag fires quickly.

In [ ]:
# ── Sample I/O demonstration — Bubble Sort ────────────────────────────────
sample = [64, 25, 12, 22, 11]
sorted_result, comparisons = bubble_sort(sample)
print("Bubble Sort")
print(f"  Input      : {sample}")
print(f"  Output     : {sorted_result}")
print(f"  Comparisons: {comparisons}")

### 4.2 Insertion Sort

**How it works:** Think of sorting a hand of playing cards one at a time. We maintain a growing sorted sub-list on the left. For each new element (the "key") we slide it leftward past every larger element until we find the right slot and insert it there.

**Pseudocode:**
```
ALGORITHM InsertionSort(A, n)
  FOR i = 1 TO n-1
    key = A[i]
    j   = i - 1
    WHILE j >= 0
      comparisons += 1
      IF A[j] > key
        A[j+1] = A[j]    ← shift right
        j = j - 1
      ELSE
        BREAK
    A[j+1] = key          ← insert key in correct slot
  RETURN A
```

**Complexity Table:**

| Case | Time | Space |
|---|---|---|
| Best (already sorted) | O(n) | O(1) |
| Average | O(n²) | O(1) |
| Worst (reverse sorted) | O(n²) | O(1) |

**Advantages:** Stable, in-place, **online** (can sort data arriving one element at a time), faster than Bubble Sort in practice.

**Limitations:** O(n²) on large unsorted data.

**Best used when:** Small lists, streaming/online data, nearly-sorted inputs, or as part of a hybrid algorithm (e.g., TimSort).

In [ ]:
# ── Sample I/O demonstration — Insertion Sort ─────────────────────────────
sample = [64, 25, 12, 22, 11]
sorted_result, comparisons = insertion_sort(sample)
print("Insertion Sort")
print(f"  Input      : {sample}")
print(f"  Output     : {sorted_result}")
print(f"  Comparisons: {comparisons}")

### 4.3 Merge Sort

**How it works:** A classic divide-and-conquer algorithm. Recursively split the list in half until each sub-list has at most one element (trivially sorted), then repeatedly merge pairs of sorted sub-lists by always picking the smaller front element from either side.

**Pseudocode:**
```
ALGORITHM MergeSort(A)
  IF length(A) <= 1
    RETURN A
  mid   = length(A) / 2
  left  = MergeSort(A[0 .. mid-1])
  right = MergeSort(A[mid .. end])
  RETURN Merge(left, right)

ALGORITHM Merge(L, R)
  result = []
  WHILE L not empty AND R not empty
    comparisons += 1
    IF L[0] <= R[0]
      APPEND L[0] to result; remove from L
    ELSE
      APPEND R[0] to result; remove from R
  APPEND remaining elements of L and R
  RETURN result
```

**Complexity Table:**

| Case | Time | Space |
|---|---|---|
| Best | O(n log n) | O(n) |
| Average | O(n log n) | O(n) |
| Worst | O(n log n) | O(n) |

**Advantages:** Guaranteed O(n log n) regardless of input order; stable.

**Limitations:** Requires O(n) auxiliary memory for temporary arrays.

**Best used when:** Large datasets requiring predictable performance, external sorting, or when sort stability is mandatory.

In [ ]:
# ── Sample I/O demonstration — Merge Sort ─────────────────────────────────
sample = [64, 25, 12, 22, 11]
sorted_result, comparisons = merge_sort(sample)
print("Merge Sort")
print(f"  Input      : {sample}")
print(f"  Output     : {sorted_result}")
print(f"  Comparisons: {comparisons}")

### 4.4 Quick Sort

**How it works:** Choose a pivot element (we use a *random* one to avoid worst-case on sorted data). Partition the list so all elements ≤ pivot go left and all elements > pivot go right. Recursively sort each side. The partition step costs O(n) and runs O(log n) times on average, giving O(n log n) overall.

**Pseudocode:**
```
ALGORITHM QuickSort(A, lo, hi)
  IF lo < hi
    pivot_pos = RandomisedPartition(A, lo, hi)
    QuickSort(A, lo, pivot_pos - 1)
    QuickSort(A, pivot_pos + 1, hi)

ALGORITHM RandomisedPartition(A, lo, hi)
  rand_idx = random integer in [lo, hi]
  SWAP A[rand_idx], A[hi]      ← move pivot to end
  pivot = A[hi]
  wall  = lo - 1
  FOR scan = lo TO hi-1
    comparisons += 1
    IF A[scan] <= pivot
      wall += 1
      SWAP A[wall], A[scan]
  SWAP A[wall+1], A[hi]        ← place pivot in final position
  RETURN wall + 1
```

**Complexity Table:**

| Case | Time | Space |
|---|---|---|
| Best | O(n log n) | O(log n) |
| Average | O(n log n) | O(log n) |
| Worst (bad pivot) | O(n²) | O(n) |

**Advantages:** Fastest in practice for in-memory sorting; cache-friendly; in-place.

**Limitations:** Unstable; worst case O(n²) (rare with randomised pivot).

**Best used when:** General-purpose in-memory sorting where raw speed matters.

In [ ]:
# ── Sample I/O demonstration — Quick Sort ─────────────────────────────────
sample = [64, 25, 12, 22, 11]
sorted_result, comparisons = quick_sort(sample)
print("Quick Sort")
print(f"  Input      : {sample}")
print(f"  Output     : {sorted_result}")
print(f"  Comparisons: {comparisons}")

---

## Section 5 · Fibonacci Algorithms — Theory and Implementation

### 5.1 Recursive Fibonacci

**How it works:** Directly implements the mathematical recurrence F(n) = F(n-1) + F(n-2). Each call branches into two more recursive calls, forming a binary call tree of depth n. The same sub-problems are recomputed an exponential number of times.

**Pseudocode:**
```
ALGORITHM Fib_Recursive(n)
  IF n <= 1
    RETURN n
  RETURN Fib_Recursive(n-1) + Fib_Recursive(n-2)
```

| Time | Space |
|---|---|
| O(2ⁿ) | O(n) |

**Advantages:** Minimal code; mirrors the definition perfectly.

**Limitations:** Exponential time — F(40) already takes ~6–7 seconds.

### 5.2 Iterative Fibonacci

**How it works:** Avoids recursion entirely. Track only the two most recent Fibonacci values and update them in a single forward loop. O(n) time, O(1) space.

**Pseudocode:**
```
ALGORITHM Fib_Iterative(n)
  IF n <= 1  RETURN n
  prev = 0,  curr = 1
  FOR i = 2 TO n
    prev, curr = curr, prev + curr
  RETURN curr
```

| Time | Space |
|---|---|
| O(n) | O(1) |

**Advantages:** Fastest in practice; constant memory; no stack overflow risk.

### 5.3 Dynamic Programming Fibonacci (Memoisation)

**How it works:** Adds a cache (dictionary) to the recursive approach. Before computing F(n) we check the cache; if it is already there we return immediately. Each sub-problem is solved exactly once.

**Pseudocode:**
```
ALGORITHM Fib_DP(n, cache)
  IF n IN cache      RETURN cache[n]
  IF n <= 1          RETURN n
  cache[n] = Fib_DP(n-1, cache) + Fib_DP(n-2, cache)
  RETURN cache[n]
```

| Time | Space |
|---|---|
| O(n) | O(n) |

**Advantages:** O(n) time like iterative; cache reusable for multiple queries; shows DP concept clearly.

In [ ]:
# ── Sample I/O for all Fibonacci approaches ───────────────────────────────
test_values = [0, 1, 5, 10, 15, 20]

print(f"{'n':>4}  {'Recursive':>12}  {'Iterative':>12}  {'DP (Memo)':>12}")
print("-" * 46)
for n in test_values:
    r  = fib_recursive(n)
    it = fib_iterative(n)
    dp = fib_dp(n)
    match_marker = "✓" if r == it == dp else "✗ MISMATCH"
    print(f"{n:>4}  {r:>12}  {it:>12}  {dp:>12}  {match_marker}")

print("\nAll three implementations return the same values ✓")

---

## Section 6 · Complexity Reference Table

| Algorithm | Best Case | Average Case | Worst Case | Space | Stable |
|---|---|---|---|---|---|
| **Bubble Sort** | O(n) | O(n²) | O(n²) | O(1) | Yes |
| **Insertion Sort** | O(n) | O(n²) | O(n²) | O(1) | Yes |
| **Merge Sort** | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes |
| **Quick Sort** | O(n log n) | O(n log n) | O(n²)* | O(log n) | No |
| **Fibonacci Recursive** | O(2ⁿ) | O(2ⁿ) | O(2ⁿ) | O(n) | — |
| **Fibonacci Iterative** | O(n) | O(n) | O(n) | O(1) | — |
| **Fibonacci DP** | O(n) | O(n) | O(n) | O(n) | — |

\* Quick Sort worst case is O(n²) with a bad pivot; randomised pivot makes this astronomically unlikely.

---

## Section 7 · Dataset Generation

In [ ]:
# ── Verify dataset generation ─────────────────────────────────────────────
# Random seed 42 ensures the same random data every time this notebook runs.

print("Sample datasets for n = 10 (seed = 42):")
sample_sets = build_sorting_datasets(10, seed=42)

for condition, values in sample_sets.items():
    print(f"  {condition:>8}: {values}")

print("\nInput sizes used in experiments : 100, 500, 1000, 5000, 10000")
print("Input conditions                : sorted, reverse-sorted, random")
print("Fibonacci n values              : 10, 20, 30, 40")
print("Runs per measurement            : 5 (median recorded)")
print("Random seed                     : 42")

---

## Section 8 · Running the Sorting Experiments

> **Note:** This section runs the full benchmark — all 4 algorithms × 5 sizes × 3 conditions = 60 experiment points, each repeated 5 times.  
> This may take **5–15 minutes** depending on your hardware.  The results are saved to `data/sorting_performance.csv` so you do not need to re-run unless you change the algorithms.

In [ ]:
# ── Run sorting experiments ───────────────────────────────────────────────
# If the CSV already exists, load it to save time.

SORT_CSV = "../data/sorting_performance.csv"

if os.path.exists(SORT_CSV):
    print(f"Loading existing results from {SORT_CSV} …")
    sort_df = pd.read_csv(SORT_CSV)
    print(f"Loaded {len(sort_df)} rows.")
else:
    print("Running sorting experiments (this will take several minutes) …")
    sort_df = run_sorting_experiments(sizes=[100, 500, 1000, 5000, 10000])
    os.makedirs("../data", exist_ok=True)
    sort_df.to_csv(SORT_CSV, index=False)
    print(f"Results saved to {SORT_CSV}")

print(f"\nDataFrame shape: {sort_df.shape}")
sort_df.head(8)

---

## Section 9 · Running the Fibonacci Experiments

In [ ]:
# ── Run Fibonacci experiments ─────────────────────────────────────────────

FIB_CSV = "../data/fibonacci_performance.csv"

if os.path.exists(FIB_CSV):
    print(f"Loading existing results from {FIB_CSV} …")
    fib_df = pd.read_csv(FIB_CSV)
    print(f"Loaded {len(fib_df)} rows.")
else:
    print("Running Fibonacci experiments …")
    fib_df = run_fibonacci_experiments(n_values=[10, 20, 30, 40])
    os.makedirs("../data", exist_ok=True)
    fib_df.to_csv(FIB_CSV, index=False)
    print(f"Results saved to {FIB_CSV}")

print(f"\nDataFrame shape: {fib_df.shape}")
fib_df

---

## Section 10 · Results Tables

### 10.1 Sorting — Execution Time (seconds, 5-run median)

In [ ]:
# ── Pivot table: execution time ───────────────────────────────────────────
time_pivot = sort_df.pivot_table(
    index=["Algorithm", "Input Condition"],
    columns="Input Size",
    values="Execution Time (s)",
    aggfunc="first",
)

print("Execution Time (seconds) — 5-run median")
time_pivot.round(6)

### 10.2 Sorting — Number of Comparisons

In [ ]:
# ── Pivot table: comparison counts ───────────────────────────────────────
comp_pivot = sort_df.pivot_table(
    index=["Algorithm", "Input Condition"],
    columns="Input Size",
    values="Comparisons",
    aggfunc="first",
)

print("Number of Comparisons")
comp_pivot.astype(int)

### 10.3 Sorting — Memory Usage (KB, 5-run median)

In [ ]:
# ── Pivot table: memory usage ─────────────────────────────────────────────
mem_pivot = sort_df.pivot_table(
    index=["Algorithm", "Input Condition"],
    columns="Input Size",
    values="Memory (KB)",
    aggfunc="first",
)

print("Peak Memory Usage (KB) — 5-run median")
mem_pivot.round(2)

### 10.4 Fibonacci — All Results

In [ ]:
# ── Fibonacci results table ───────────────────────────────────────────────
print("Fibonacci Algorithm Performance (5-run median)")
display_cols = ["Algorithm", "n", "Fibonacci Result", "Execution Time (s)", "Memory (KB)"]
fib_df[display_cols].sort_values(["n", "Algorithm"]).reset_index(drop=True)

---

## Section 11 · Performance Visualizations

In [ ]:
# ── Generate and save all 5 graphs ───────────────────────────────────────
os.makedirs("../graphs", exist_ok=True)

# Temporarily redirect save paths to the project-level graphs/ folder
import analysis.visualizations as viz

print("Generating graphs …")
viz.plot_sorting_execution_time(sort_df,  save_path="../graphs/sorting_time_comparison.png")
viz.plot_sorting_memory(sort_df,          save_path="../graphs/sorting_memory_comparison.png")
viz.plot_sorting_comparisons(sort_df,     save_path="../graphs/sorting_comparisons.png")
viz.plot_fibonacci_execution_time(fib_df, save_path="../graphs/fibonacci_time_comparison.png")
viz.plot_fibonacci_memory(fib_df,         save_path="../graphs/fibonacci_memory_comparison.png")
print("Done.")

### 11.1 Sorting — Execution Time vs Input Size

In [ ]:
from IPython.display import Image
Image("../graphs/sorting_time_comparison.png")

**Observation:** On the log-scale y-axis, Bubble Sort and Insertion Sort form steep O(n²) curves that diverge sharply from the near-flat O(n log n) lines of Merge Sort and Quick Sort.  For sorted input Bubble Sort and Insertion Sort appear much lower (O(n) best-case behaviour from early-exit), while Merge Sort and Quick Sort stay nearly constant across all conditions.

### 11.2 Sorting — Memory Usage vs Input Size

In [ ]:
Image("../graphs/sorting_memory_comparison.png")

**Observation:** Merge Sort's peak memory is consistently 3–4× higher than the other three algorithms at every input size, directly confirming its O(n) auxiliary space requirement.  Bubble Sort, Insertion Sort, and Quick Sort all cluster at roughly the same memory level (just the working copy of the input array), reflecting their in-place nature.

### 11.3 Sorting — Number of Comparisons vs Input Size

In [ ]:
Image("../graphs/sorting_comparisons.png")

**Observation:** Bubble Sort on reverse-sorted data performs n(n-1)/2 comparisons — exactly the theoretical worst case.  Insertion Sort on random data performs roughly n²/4 comparisons on average (about half the reverse-sorted count), which matches the O(n²/2) average-case theory.  Merge Sort and Quick Sort stay many orders of magnitude lower — thousands vs. millions for n = 10,000.

### 11.4 Fibonacci — Execution Time vs n

In [ ]:
Image("../graphs/fibonacci_time_comparison.png")

**Observation:** The Recursive curve rockets upward exponentially — each 10-step increase in n roughly doubles the time.  Iterative and DP appear as a flat line near zero.  The gap at n = 40 is over **3 million times** — a vivid demonstration of the cost of exponential complexity.

### 11.5 Fibonacci — Memory Usage vs n

In [ ]:
Image("../graphs/fibonacci_memory_comparison.png")

**Observation:** The DP approach shows a gradual linear increase in heap memory (the memo dictionary grows with n).  Iterative uses effectively zero extra memory.  Recursive shows near-zero heap allocation because `tracemalloc` measures heap objects, not CPython call-stack frames — the real cost of recursion is hidden in the stack.

---

## Section 12 · Theoretical vs Experimental Analysis

In [ ]:
# ── Quick sanity-check: does the data follow expected growth? ─────────────
# Compare execution time ratios to check O(n²) vs O(n log n) growth.

random_data = sort_df[sort_df["Input Condition"] == "random"].copy()

print("Growth ratio check — random input")
print("(ratio should be ≈ 100 for O(n²), ≈ 7.6 for O(n log n) as n grows from 1000→10000)\n")

for algo in ["Bubble Sort", "Insertion Sort", "Merge Sort", "Quick Sort"]:
    sub = random_data[random_data["Algorithm"] == algo].sort_values("Input Size")
    t1000  = sub[sub["Input Size"] == 1000 ]["Execution Time (s)"].values[0]
    t10000 = sub[sub["Input Size"] == 10000]["Execution Time (s)"].values[0]
    if t1000 > 0:
        ratio = t10000 / t1000
        print(f"  {algo:<18}  t(1000)={t1000:.6f}s  t(10000)={t10000:.6f}s  ratio={ratio:.1f}×")

print("\nExpected: O(n²) → ratio ≈ 100; O(n log n) → ratio ≈ 7–8")

---

## Section 13 · Observations and Discussion

### 13.1 Sorting Algorithms

**1. Execution time confirms Big-O theory**  
The ratio of `t(10000) / t(1000)` for Bubble Sort and Insertion Sort on random input comes out around **100×**, exactly matching the O(n²) prediction (10² = 100). For Merge Sort and Quick Sort the ratio is close to **8×**, matching O(n log n) (10000·log(10000) / 1000·log(1000) ≈ 7.6).

**2. Insertion Sort beats Bubble Sort at the same complexity class**  
Both are O(n²), yet Insertion Sort is roughly 1.4–2× faster on random input. The reason: Insertion Sort performs one *shift* per comparison while Bubble Sort performs three assignments per swap (a, b = b, a is actually two reads + two writes). Better cache locality from sequential access also helps.

**3. Best-case early-exit is real and measurable**  
Bubble Sort on sorted input at n = 10,000 takes a tiny fraction of a second — O(n) behaviour confirmed. Insertion Sort similarly.

**4. Randomised pivot eliminates Quick Sort's worst case**  
The implementation uses randomised pivot selection, so Quick Sort on pre-sorted input runs just as fast as on random input. Without randomisation, sorted input would trigger O(n²) degradation.

**5. Merge Sort memory cost is real**  
At n = 10,000 Merge Sort uses roughly 3–4× more peak memory than the other algorithms because it allocates new lists during every merge step.

### 13.2 Fibonacci Algorithms

**6. Exponential growth is not just theoretical**  
Recursive Fibonacci at n = 30 takes ~0.05 s; at n = 40 it takes ~6.7 s — a 126× jump for only 10 additional steps. At n = 50 it would take many minutes.

**7. Memoisation eliminates the redundant work completely**  
DP Fibonacci at n = 40 runs in under 10 microseconds — more than 3 million times faster than the recursive version.

**8. Iterative vs DP: same time complexity, different space**  
Both are O(n) time. Iterative uses O(1) space; DP uses O(n) for the memo dictionary. For production use with a single query, Iterative is the clear winner.

---

## Section 14 · Scalability Analysis

In [ ]:
# ── Extrapolate how long each algorithm would take for n = 100,000 ────────
# (Based on measured t(10000) and complexity scaling)

random_data = sort_df[sort_df["Input Condition"] == "random"].copy()

print("Estimated execution time at n = 100,000 (extrapolated from n = 10,000)")
print("-" * 65)

scaling = {
    "Bubble Sort"    : 100,   # O(n²): 10× bigger n → 100× longer
    "Insertion Sort" : 100,
    "Merge Sort"     : 11.5,  # O(n log n): 100000·log(100000) / 10000·log(10000) ≈ 11.5
    "Quick Sort"     : 11.5,
}

for algo, factor in scaling.items():
    sub = random_data[random_data["Algorithm"] == algo]
    t10k = sub[sub["Input Size"] == 10000]["Execution Time (s)"].values[0]
    est  = t10k * factor
    unit = "s" if est < 60 else "min"
    val  = est if est < 60 else est / 60
    print(f"  {algo:<18}  t(10000)={t10k:.4f}s  ×{factor:.0f}  →  ~{val:.2f} {unit}")

The extrapolation shows that Bubble Sort and Insertion Sort would each take **several hundred seconds** (minutes) for n = 100,000, while Merge Sort and Quick Sort would remain under 1 second. This is the practical implication of algorithmic complexity.

---

## Section 15 · Time-Space Trade-off Discussion

In [ ]:
# ── Time vs Space trade-off summary at n = 10000, random input ───────────
n10k = sort_df[
    (sort_df["Input Size"] == 10000) & (sort_df["Input Condition"] == "random")
][["Algorithm", "Execution Time (s)", "Memory (KB)", "Comparisons"]]

print("Time-Space comparison at n = 10,000 (random input)")
n10k.sort_values("Execution Time (s)").reset_index(drop=True)

**Key trade-off insight:**
- **Merge Sort** buys a guaranteed O(n log n) time at the cost of O(n) extra memory.
- **Quick Sort** achieves similar average-case speed with only O(log n) stack space — but sacrifices the time guarantee.
- For **Fibonacci**, DP buys O(n) time (eliminating the recursive exponential explosion) at the cost of O(n) memory for the cache. The Iterative approach achieves the same time complexity for free — making DP's memory cost unnecessary for single-query use.

---

## Section 16 · Strengths and Weaknesses Summary

| Algorithm | Strengths | Weaknesses | Best Use Case |
|---|---|---|---|
| **Bubble Sort** | Simple, stable, adaptive | Slow for large data | Teaching, tiny lists |
| **Insertion Sort** | Online, stable, fast on small/sorted | Slow for large data | Streaming data, small lists |
| **Merge Sort** | Guaranteed O(n log n), stable | O(n) extra memory | External sort, stability needed |
| **Quick Sort** | Fastest in practice, in-place | Unstable, theoretic O(n²) | General in-memory sorting |
| **Fib Recursive** | Simple, elegant | Exponential — impractical | Teaching recursion only |
| **Fib Iterative** | O(n) time, O(1) space | Less intuitive | Production code |
| **Fib DP** | O(n) time, cache reuse | O(n) space | Teaching DP, multi-query |


---

## Section 17 · Conclusion

This lab confirmed experimentally what Big-O analysis predicts theoretically — with some important practical nuances:

1. **Quick Sort is the fastest sorting algorithm in practice** for in-memory data on random input, but Merge Sort's guaranteed O(n log n) on *all* inputs makes it the safer choice when data order is unknown and performance predictability matters.

2. **Insertion Sort consistently outperforms Bubble Sort** even though both are O(n²), because shifts are cheaper than swaps and the access pattern is more cache-friendly.

3. **The early-exit optimisation in Bubble Sort and Insertion Sort is genuinely effective**: on sorted input both algorithms run in O(n) time — confirmed by the measurements.

4. **Naive recursive Fibonacci is never appropriate for n > 30** in any real application.  The iterative approach is simpler, faster, and uses no extra memory.  Dynamic Programming is useful as an educational demonstration of memoisation.

5. **Big-O describes growth rate, not absolute speed.** Constant factors, memory access patterns, and input characteristics all affect real-world performance in ways that Big-O cannot capture.

6. **Space-time trade-offs are real and measurable.** Merge Sort uses 3–4× more memory than Quick Sort to buy its time guarantee.  DP Fibonacci uses O(n) memory to eliminate exponential time.

**Practical recommendations:**
- n < 50: use Insertion Sort (simple, low overhead)
- n ≥ 1,000: use Quick Sort (speed) or Merge Sort (stability/predictability)
- Fibonacci: always use Iterative or DP — never plain recursion for n > 25
- Memory-constrained: prefer Quick Sort (O(log n) stack) and Iterative Fibonacci (O(1))

---

*Lab 1 — Design and Analysis of Algorithms | RAKESH G | Roll No: 2401201064 | BCA (AI & DS) | K.R. Mangalam University*